In [1]:
import os
os.environ["GOOGLE_GEMINI_KEY"]="#add the API Key here"

In [10]:
!pip install -qU youtube-transcript-api langchain-community langchain-faiss langchain-text-splitters langchain-google-genai \
               faiss-cpu tiktoken python-dotenv

In [13]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

##step 1a- Indexing (Document Ingestion)


In [27]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled

video_id = "tCMzkJxBM5c"

try:
    fetched_transcript = YouTubeTranscriptApi().fetch(video_id, languages=['en'])
    transcript_list = fetched_transcript.to_raw_data()
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")
except Exception as e:
    print(f"An error occurred: {e}")

Larry Fink, the guy running BlackRock, >> [music] >> is really the president of the United States if we look at the kind of influence he's got. Wait a minute. These guys essentially have a monopoly. Well, how big is BlackRock? >> Ladies and gentlemen, BlackRock is the most powerful, and according to US politicians, the most evil company in the world. >> [music] >> BlackRock, the largest asset manager in the world. >> [music] >> He can get a meeting with any company on the planet. >> [music] >> Jensen Huang built Nvidia from scratch, and today he owns 3.7% of Nvidia. But BlackRock, BlackRock owns 8.02%. Steve Jobs owns less than 1% of Apple, but BlackRock owns 7.97% of Apple. Bill Gates owns 1.34% of Microsoft, but BlackRock owns 8.18% of Microsoft. And just like this, the big three, which includes BlackRock, State Street, and Vanguard, they own a huge stake in almost every S&P 500 company. And BlackRock alone controls over 15 trillion dollars in assets. Do you realize how much 15 trill

In [29]:
len(transcript_list )

574

##step 1b - Indexing (Text Splitting)

In [32]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.create_documents([transcript])

In [33]:
len(chunks)

51

In [34]:
chunks[0]

Document(metadata={}, page_content="Larry Fink, the guy running BlackRock, >> [music] >> is really the president of the United States if we look at the kind of influence he's got. Wait a minute. These guys essentially have a monopoly. Well, how big is BlackRock? >> Ladies and gentlemen, BlackRock is the most powerful, and according to US politicians, the most evil company in the world. >> [music] >> BlackRock, the largest asset manager in the world. >> [music] >> He can get a meeting with any company on the planet. >> [music] >>")

##step 1c -- Chunks to vectors ( Embedding generation and storing in vector store)

In [40]:
import os

# Retrieve the API key from the correct environment variable name
gemini_key = os.environ.get("GOOGLE_GEMINI_KEY")

# Pass the key explicitly to the embeddings model
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", google_api_key=gemini_key)

# Initialize the vector store
vector_store = FAISS.from_documents(chunks, embeddings)

In [41]:
vector_store.index_to_docstore_id

{0: 'd9be57eb-576c-4e5f-a021-5a17b1d3901b',
 1: '9d951db3-a30b-49ac-980b-a8e1345384d8',
 2: '711d6bb1-e37e-4e84-a3e3-82cc77f7a8dd',
 3: 'dc72c25b-98f1-4a84-8f4f-74dff9a89814',
 4: '0e7b3d7e-71e7-47cb-806c-3910d70d1445',
 5: '2d66c1d3-7242-453e-af4c-957bddba2399',
 6: 'b38e9619-27b7-45eb-8434-bd2e3f77cb92',
 7: 'a46759c8-6c79-4a1f-81ee-50ff3b864ced',
 8: 'fb6f4580-85a6-4f40-9c1c-e835262d1a28',
 9: '197f77a6-302b-4459-83e5-a39c23b58d26',
 10: '7818fd50-4737-4c46-9c38-79fd896ab698',
 11: '33ee8fac-3f24-4c0f-8dba-d8541d0011e0',
 12: '843d86f2-dd98-4fca-8410-7e217f45baad',
 13: 'ec063b36-8274-486c-94cb-af51714d37c3',
 14: 'dfd6b24f-898f-4125-8693-0c29521338df',
 15: 'a3da3923-eb34-4e9b-b822-0633df7145c1',
 16: '78f14b8c-c4d6-4c4e-9031-b4f9c6cbe5ee',
 17: 'f21d692d-41fd-4644-996e-05ab07091f73',
 18: '800d8e89-e320-465f-b433-6f000185249f',
 19: '40e47520-aae3-41fe-8660-e909607b8d21',
 20: 'fba8b228-ad5e-4bdb-a5e9-7af5976914c1',
 21: 'd5e1a0d6-9954-454d-8a0a-6c1721b35232',
 22: '12032133-1a9a-

In [42]:
vector_store.get_by_ids(['ec063b36-8274-486c-94cb-af51714d37c3'])

[Document(id='ec063b36-8274-486c-94cb-af51714d37c3', metadata={}, page_content='[music] this is where Larry Fink designed a genius strategy. He cut this asset into three parts. So, instead of selling everyone an equal slice of the loans, >> [music] >> he sold three bonds with a payment queue. For example, bond A has a size of $50 million, but only paid 6% interest. Bond B had a size of $30 million and paid 7% interest. And lastly, bond C had a size of $20 million paying 8% interest. >> [music] >> So, now every time a homeowner pays their principal, the money first goes to')]

###Step 2 -> Retreiver --> query --> embed--> vector--> then search in to vector_store

In [52]:
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 4})

In [54]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7ce554ece660>, search_kwargs={'k': 4})

In [55]:
retriever.invoke("How does the blackrock works")

[Document(id='d5e1a0d6-9954-454d-8a0a-6c1721b35232', metadata={}, page_content="inside Blackstone Group's office in New York and called it Blackstone Financial Management, later to be renamed as BlackRock. And they built a magical risk machine in BlackRock. You know what they did? They kept on collecting as much data as they possibly can and as [music] many variables as they possibly can. This included something as simple as an oil crash all the way to wars and even tsunamis. And they ran a program to understand how each of these disasters will affect bonds, loans, credit"),
 Document(id='8ced0eec-aa76-441c-a1e3-4c2d47008290', metadata={}, page_content='trillion with a 0.03% commission running on autopilot. That is how BlackRock became a money-making machine. And every dollar invested in their S&P 500 bought them a stake in Nvidia, Apple, Microsoft, and 497 [music] of the most powerful companies on Earth. That is how BlackRock today owns 8.18% of Microsoft, 8.02% of Nvidia, 6.69% [musi

**Step 3 - Augumentation**

In [65]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", google_api_key=gemini_key, temperature=0.2)

In [49]:
prompt= PromptTemplate(
    template="""
    You are a helpful assistant.
    Answer ONLY from the provided transcript context.
    If the context is insufficient, just say get the f*ck out of here, I don't know.

    {context}
    Question: {question}
    """,
    input_variables=["context", "question"]
)

In [78]:
question       = "does this topic of Indian Employees discussed in the video?"
retrieved_docs= retriever.invoke(question)

In [79]:
retrieved_docs

[Document(id='d1a7dc26-5f38-4cb6-be0b-15968734fc16', metadata={}, page_content="that is already the case, and I just hope this does not happen in India. This is the dark truth of capitalism that we all need to learn and ensure that our country never gets into. That's all for my side today, guys. If you learned something valuable, please make sure to hit the like button to support our work. And for more such insightful political and business case studies, please subscribe to our channel. Thank you so much for watching. I will see you in the next one. Bye-bye. >> [music]"),
 Document(id='2d66c1d3-7242-453e-af4c-957bddba2399', metadata={}, page_content="here in India. And here's why every Indian investor needs to hear this. In 2010, $1 cost you 45 rupees. Today, it's over 90. So, even when your Indian portfolio is doing well in rupees, your global purchasing power is silently leaking every single year. And it's not just currency. The US and Indian markets move on completely different cycl

In [80]:
#mere pass ab 4 documents a regarding the blackrock asset info

content_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
content_text

"that is already the case, and I just hope this does not happen in India. This is the dark truth of capitalism that we all need to learn and ensure that our country never gets into. That's all for my side today, guys. If you learned something valuable, please make sure to hit the like button to support our work. And for more such insightful political and business case studies, please subscribe to our channel. Thank you so much for watching. I will see you in the next one. Bye-bye. >> [music]\n\nhere in India. And here's why every Indian investor needs to hear this. In 2010, $1 cost you 45 rupees. Today, it's over 90. So, even when your Indian portfolio is doing well in rupees, your global purchasing power is silently leaking every single year. And it's not just currency. The US and Indian markets move on completely different cycles. Some years India wins, some years US wins. So, betting your entire portfolio on just one economy is a risk that most people don't even realize they're\n\ni

In [81]:
final_prompt = prompt.invoke({"context": content_text, "question" : question})

STEP 4 **Generation**

In [82]:
answer= llm.invoke(final_prompt)
print(answer.content)

[{'type': 'text', 'text': "get the f*ck out of here, I don't know.", 'extras': {'signature': 'EnMKcQFpFH0TabYPK4ZfVHMn7A3BXKEeIFAkMHteXQXZWZBTvS8IKUbCO9J6zklwUi6CHpYqw8JmWIiAsDSQo2ZuE0kbPiP4E1VjKYpFoVRPJfDikYvuIdgR9ZYahy2P+f8T8Ldh2vTjZyoBdguvdWHswqxS'}}]


#**Building** A **CHAIN**

In [83]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda,RunnableParallel
from langchain_core.output_parsers import StrOutputParser

In [84]:
def format_docs(retrieved_docs):
  content_text="\n\n".join(doc.page_content for doc in retrieved_docs)
  return content_text

In [92]:
paraller_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [93]:
paraller_chain.invoke("What is Blackrock & Who is Larry?")

{'context': "Larry Fink, the guy running BlackRock, >> [music] >> is really the president of the United States if we look at the kind of influence he's got. Wait a minute. These guys essentially have a monopoly. Well, how big is BlackRock? >> Ladies and gentlemen, BlackRock is the most powerful, and according to US politicians, the most evil company in the world. >> [music] >> BlackRock, the largest asset manager in the world. >> [music] >> He can get a meeting with any company on the planet. >> [music] >>\n\nmade a physical product in their existence. But even then, a failed banker somehow turned it into the most powerful company on Earth. So powerful that the US government itself is now afraid of BlackRock. >> BlackRock is an extremely dangerous company. [music] >> What Larry Fink and BlackRock have done, they are trying to suppress investment in the fossil fuel industry in America. >> The question is, how did a company that make nothing end up owning everything? Why is the US govern

In [87]:
parser= StrOutputParser()

In [95]:
#know we will be making main chain having 1st two paraller chain then to next chain of  prompt-> LLM --> Parser
main_chain = paraller_chain | prompt | llm | parser


In [96]:
main_chain.invoke("What is Blackrock & Who is Larry?")

'BlackRock is the largest asset manager in the world and a company that makes nothing. Larry Fink is the guy running BlackRock and a failed banker who turned it into the most powerful company on Earth.'